#Importing necessary libraries


In [1]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv(
"https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv"
)

# Preprocessing

In [3]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [5]:
df.drop(['id','Unnamed: 32'],inplace = True,axis = 1)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    object 
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_se                  5

In [7]:
x_train,x_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size = 0.2)

In [8]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [9]:
y_train

,diagnosis
278,B
498,M
67,B
488,B
60,B
...,...
448,B
183,B
151,B
493,B


In [10]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [11]:
y_train = y_train.astype(float)
y_test = y_test.astype(float)

# Numpy arrays to PyTorch tensors

In [12]:
x_train_tensor = torch.from_numpy(x_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

# Neural Network

In [19]:
class MySimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features,1),
        nn.Sigmoid()
    )

  def forward(self,features):
    out = self.network(features)
    return out

# Training Pipeline

In [14]:
learning_rate = 0.1
epochs = 25

In [20]:
#loss function
loss_function = nn.BCELoss()

In [25]:
y_train_tensor.shape

torch.Size([455])

In [29]:
model = MySimpleNN(x_train_tensor.shape[1])

#optimizer
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

#define loop
for epoch in range(epochs):
  y_pred = model(x_train_tensor.float())#need to use float32 by default but we converted arrays to tensors so dtype will be float64 but pytorch uses float32,so we need to change
  loss = loss_function(y_pred, y_train_tensor.float().reshape(-1, 1))

  #clear gradients
  optimizer.zero_grad()

  #backward pass
  loss.backward()

  #parameters update
  optimizer.step()

  #print loss
  print(f"epoch: {epoch+1},loss :{loss}")

epoch: 1,loss :0.6390606164932251
epoch: 2,loss :0.4841362535953522
epoch: 3,loss :0.4025857448577881
epoch: 4,loss :0.3522005081176758
epoch: 5,loss :0.31751346588134766
epoch: 6,loss :0.2918784022331238
epoch: 7,loss :0.2719772458076477
epoch: 8,loss :0.25596192479133606
epoch: 9,loss :0.24271684885025024
epoch: 10,loss :0.23152576386928558
epoch: 11,loss :0.2219061702489853
epoch: 12,loss :0.2135201245546341
epoch: 13,loss :0.20612314343452454
epoch: 14,loss :0.19953373074531555
epoch: 15,loss :0.19361382722854614
epoch: 16,loss :0.18825645744800568
epoch: 17,loss :0.18337728083133698
epoch: 18,loss :0.17890873551368713
epoch: 19,loss :0.17479604482650757
epoch: 20,loss :0.17099425196647644
epoch: 21,loss :0.16746602952480316
epoch: 22,loss :0.16418009996414185
epoch: 23,loss :0.161110058426857
epoch: 24,loss :0.15823335945606232
epoch: 25,loss :0.15553069114685059


In [27]:
y_pred.shape

torch.Size([455, 1])

In [16]:
model.network[0].weight

Parameter containing:
tensor([[-0.1382,  0.1372, -0.1420,  0.1634,  0.1055,  0.0643, -0.0628,  0.0551,
         -0.0132, -0.1292,  0.0767, -0.0011,  0.1255, -0.0372, -0.1494, -0.0986,
          0.0851,  0.0185, -0.1037,  0.0979, -0.0077, -0.1364,  0.0445, -0.0626,
          0.1040, -0.1164,  0.0566, -0.1600,  0.0037,  0.1048]],
       requires_grad=True)

In [17]:
model.network[0].bias

Parameter containing:
tensor([-0.1155], requires_grad=True)

In [18]:
with torch.no_grad():
  y_pred = model.forward(x_test_tensor.float())
  y_pred = (y_pred > 0.5).float()

print(y_pred)



tensor([[0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
      